# Complaint Theme Analysis

## Project Overview
Analyze customer complaint data to identify recurring complaint themes, affected products, escalation patterns, and business risks. This project uses keyword-based theme extraction and pattern analysis to derive actionable insights.

## Learning Objectives
- Categorize complaints into themes using keyword matching
- Analyze complaint volume, severity, and resolution patterns
- Identify high-risk products and escalation drivers
- Generate prioritized recommendations for complaint reduction

## Problem Statement
A company receives hundreds of customer complaints monthly. Without systematic analysis, complaints are handled reactively. The goal is to identify recurring themes, products with the most complaints, and patterns that indicate systemic issues.

## Why This Project Matters
Unresolved complaints lead to churn, negative reviews, and brand damage. Identifying complaint themes enables proactive fixes — addressing the root cause of 10 complaints is more efficient than solving them individually.

## Dataset Overview
Synthetic customer complaint data: ~4K complaints over 2 years with complaint text, category, product, severity, escalation status, resolution time, and channel.

## Dataset Source and License Notes
- Synthetic data generated for educational purposes
- Patterns inspired by common customer service complaint patterns
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))

# Theme keyword dictionaries
THEME_KEYWORDS = {
    'Billing / Charges': ['billing','charge','overcharge','refund','invoice','payment','fee','subscription','charged','price'],
    'Product Quality': ['defective','broken','quality','damaged','faulty','crack','malfunction','stopped working','poor quality'],
    'Shipping / Delivery': ['shipping','delivery','late','delayed','lost','package','tracking','arrived','missing','courier'],
    'Customer Service': ['rude','unhelpful','wait','hold','transfer','agent','support','response','ignored','callback'],
    'Website / App': ['website','app','login','error','crash','slow','bug','glitch','page','checkout'],
    'Return / Exchange': ['return','exchange','refund policy','send back','replacement','warranty','repair'],
}
print('Configuration set')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_complaints = 4000

products = ['Laptop Pro X', 'Wireless Earbuds', 'Smart Watch', 'Fitness Tracker',
            'Bluetooth Speaker', 'Phone Case', 'Charger Cable', 'Screen Protector',
            'Tablet Stand', 'Webcam HD']

channels = ['Email', 'Phone', 'Live Chat', 'Social Media', 'In-Store']
channel_weights = [0.30, 0.25, 0.25, 0.12, 0.08]

severity_levels = ['Low', 'Medium', 'High', 'Critical']

# Complaint templates by theme
templates = {
    'Billing / Charges': [
        'I was overcharged on my billing statement for {product}.',
        'My invoice shows a charge I did not authorize for {product}.',
        'I need a refund for the duplicate payment on {product}.',
        'The subscription fee for {product} was charged after I cancelled.',
        'Incorrect price charged at checkout for {product}.',
    ],
    'Product Quality': [
        '{product} arrived defective and stopped working after 2 days.',
        'The {product} has poor quality build, cracked on first use.',
        '{product} is broken out of the box, clearly a malfunction.',
        'Quality of {product} is terrible, very disappointed.',
        '{product} damaged during normal use, faulty design.',
    ],
    'Shipping / Delivery': [
        'My {product} delivery was delayed by over a week.',
        'Package for {product} shows delivered but never arrived, lost.',
        'Shipping tracking for {product} has not updated in 5 days.',
        '{product} arrived late and the package was damaged.',
        'Missing items in my {product} shipment, courier issue.',
    ],
    'Customer Service': [
        'Agent was rude and unhelpful when I called about {product}.',
        'Put on hold for 45 minutes, then call was transferred twice about {product}.',
        'No response to my support ticket about {product} for 2 weeks.',
        'Customer service ignored my complaint about {product}.',
        'Requested a callback about {product} but never received one.',
    ],
    'Website / App': [
        'Website checkout crashed when ordering {product}.',
        'App login error prevents me from tracking my {product} order.',
        'The website is incredibly slow when browsing {product} page.',
        'Got a glitch on the app trying to return {product}.',
        'Checkout page error when applying coupon for {product}.',
    ],
    'Return / Exchange': [
        'Trying to return {product} but refund policy is unclear.',
        'Exchange request for {product} has been pending for 3 weeks.',
        'Warranty claim for {product} was denied without explanation.',
        'Need a replacement for {product}, current one is defective.',
        'Cannot find how to send back {product} for repair.',
    ],
}

themes = list(templates.keys())
theme_weights = [0.22, 0.25, 0.18, 0.12, 0.10, 0.13]

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')

rows = []
for i in range(n_complaints):
    theme = np.random.choice(themes, p=theme_weights)
    product = np.random.choice(products)
    text = np.random.choice(templates[theme]).format(product=product)
    date = np.random.choice(dates)
    channel = np.random.choice(channels, p=channel_weights)

    # Severity correlates with theme
    if theme in ['Product Quality', 'Billing / Charges']:
        sev = np.random.choice(severity_levels, p=[0.10, 0.30, 0.35, 0.25])
    elif theme == 'Customer Service':
        sev = np.random.choice(severity_levels, p=[0.15, 0.35, 0.30, 0.20])
    else:
        sev = np.random.choice(severity_levels, p=[0.25, 0.35, 0.25, 0.15])

    escalated = np.random.random() < (0.40 if sev == 'Critical' else 0.20 if sev == 'High' else 0.08)
    resolution_days = max(1, int(np.random.exponential(scale=5 if sev in ['Critical','High'] else 3)))

    rows.append({
        'ComplaintID': f'CMP{i:05d}', 'Date': date, 'Product': product,
        'Channel': channel, 'ComplaintText': text, 'ActualTheme': theme,
        'Severity': sev, 'Escalated': escalated, 'ResolutionDays': resolution_days,
    })

df = pd.DataFrame(rows)
df['Date'] = pd.to_datetime(df['Date'])
df['YearMonth'] = df['Date'].dt.to_period('M')
df = df.sort_values('Date').reset_index(drop=True)
print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values:\n{df.isnull().sum()}')
print(f'\nDate range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'Unique products: {df["Product"].nunique()}')
print(f'Escalation rate: {df["Escalated"].mean():.1%}')
print(f'Avg resolution days: {df["ResolutionDays"].mean():.1f}')
print(f'\nSeverity distribution:\n{df["Severity"].value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Complaints by theme
df['ActualTheme'].value_counts().plot.barh(ax=axes[0,0], edgecolor='black', color='steelblue')
axes[0,0].set_title('Complaints by Theme')
axes[0,0].invert_yaxis()

# Severity distribution
sev_order = ['Low', 'Medium', 'High', 'Critical']
df['Severity'].value_counts().reindex(sev_order).plot.bar(ax=axes[0,1], edgecolor='black',
    color=['#66bb6a','#ffa726','#ef5350','#b71c1c'])
axes[0,1].set_title('Severity Distribution')
axes[0,1].tick_params(axis='x', rotation=0)

# Monthly complaint volume
monthly = df.groupby('YearMonth').size()
monthly.plot(ax=axes[1,0], marker='o')
axes[1,0].set_title('Monthly Complaint Volume')
axes[1,0].tick_params(axis='x', rotation=45)

# Complaints by product
df['Product'].value_counts().plot.barh(ax=axes[1,1], edgecolor='black', color='coral')
axes[1,1].set_title('Complaints by Product')
axes[1,1].invert_yaxis()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Keyword-Based Theme Detection

In [ ]:
def detect_theme(text):
    text_lower = text.lower()
    scores = {}
    for theme, keywords in THEME_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in text_lower)
        if score > 0:
            scores[theme] = score
    if scores:
        return max(scores, key=scores.get)
    return 'Other'

df['DetectedTheme'] = df['ComplaintText'].apply(detect_theme)

# Compare detected vs actual
accuracy = (df['DetectedTheme'] == df['ActualTheme']).mean()
print(f'Theme detection accuracy: {accuracy:.1%}')
print(f'\nDetected theme distribution:\n{df["DetectedTheme"].value_counts()}')

## Theme × Severity Heatmap

In [ ]:
theme_sev = pd.crosstab(df['ActualTheme'], df['Severity'])[sev_order]
plt.figure(figsize=(10, 6))
sns.heatmap(theme_sev, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Complaint Count: Theme × Severity')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'theme_severity.png'), dpi=100, bbox_inches='tight')
plt.show()

## Escalation Analysis

In [ ]:
esc_by_theme = df.groupby('ActualTheme').agg(
    total=('Escalated', 'count'),
    escalated=('Escalated', 'sum'),
    escalation_rate=('Escalated', 'mean'),
    avg_resolution=('ResolutionDays', 'mean'),
).round(3).sort_values('escalation_rate', ascending=False)
print('Escalation by Theme:')
print(esc_by_theme)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
esc_by_theme['escalation_rate'].plot.bar(ax=axes[0], edgecolor='black', color='tomato')
axes[0].set_title('Escalation Rate by Theme')
axes[0].set_ylabel('Escalation Rate')
axes[0].tick_params(axis='x', rotation=45)

esc_by_theme['avg_resolution'].plot.bar(ax=axes[1], edgecolor='black', color='steelblue')
axes[1].set_title('Avg Resolution Days by Theme')
axes[1].set_ylabel('Days')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'escalation_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## High-Risk Products

In [ ]:
product_risk = df.groupby('Product').agg(
    total_complaints=('ComplaintID', 'count'),
    critical_count=('Severity', lambda x: (x == 'Critical').sum()),
    escalation_rate=('Escalated', 'mean'),
    avg_resolution=('ResolutionDays', 'mean'),
).round(3)
product_risk['critical_pct'] = (product_risk['critical_count'] / product_risk['total_complaints'] * 100).round(1)
product_risk = product_risk.sort_values('total_complaints', ascending=False)
print('Product Risk Summary:')
print(product_risk)

## Channel Analysis

In [ ]:
channel_stats = df.groupby('Channel').agg(
    complaints=('ComplaintID', 'count'),
    escalation_rate=('Escalated', 'mean'),
    avg_resolution=('ResolutionDays', 'mean'),
    pct_critical=('Severity', lambda x: (x == 'Critical').mean()),
).round(3).sort_values('complaints', ascending=False)
print('Channel Statistics:')
print(channel_stats)

fig, ax = plt.subplots(figsize=(8, 5))
channel_stats['complaints'].plot.bar(ax=ax, edgecolor='black', color='mediumpurple')
ax.set_title('Complaints by Channel')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'channel_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Complaint Trend by Theme

In [ ]:
theme_monthly = df.groupby(['YearMonth', 'ActualTheme']).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(14, 6))
theme_monthly.plot(ax=ax, marker='o', alpha=0.7)
ax.set_title('Monthly Complaint Volume by Theme')
ax.set_ylabel('Complaints')
ax.legend(title='Theme', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'theme_trends.png'), dpi=100, bbox_inches='tight')
plt.show()

## Business Risk Summary

In [ ]:
print('=' * 60)
print('BUSINESS RISK SUMMARY')
print('=' * 60)
critical_total = (df['Severity'] == 'Critical').sum()
esc_total = df['Escalated'].sum()
print(f'Total complaints: {len(df)}')
print(f'Critical severity: {critical_total} ({critical_total/len(df):.1%})')
print(f'Escalated: {esc_total} ({esc_total/len(df):.1%})')
print(f'Avg resolution time: {df["ResolutionDays"].mean():.1f} days')
print()
print('Top 3 riskiest themes (by escalation rate):')
for theme, row in esc_by_theme.head(3).iterrows():
    print(f'  {theme}: {row["escalation_rate"]:.1%} escalation, {row["avg_resolution"]:.1f} avg days')
print()
print('Priority actions:')
print('  1. Investigate Product Quality defects — highest volume and severity')
print('  2. Audit Billing/Charges for overcharge root causes')
print('  3. Improve Customer Service training to reduce escalations')
print('  4. Streamline Return/Exchange process to cut resolution time')

## Interpretation and Business Insight
- **Product Quality** and **Billing/Charges** are the top complaint themes with highest severity skew
- Escalation rates are highest for themes involving money or defective products — these trigger stronger customer emotions
- Resolution times are longest for complaints requiring cross-department coordination (returns, billing disputes)
- Social Media complaints have lower volume but potentially higher visibility and brand risk
- Products with high critical complaint rates warrant immediate quality review

## Limitations
- Synthetic data with pre-assigned themes — real complaints require NLP for theme extraction
- Keyword-based theme detection is simplistic; real complaints often span multiple themes
- No customer-level analysis (repeat complainers, lifetime value impact)
- Resolution time is simulated; real resolution tracking depends on CRM data quality

## How to Improve This Project
- Use real customer complaint data from a CRM export
- Apply topic modeling (LDA, BERTopic) for automated theme discovery
- Add sentiment analysis to quantify complaint intensity
- Build a complaint priority scoring model combining severity, product risk, and customer value
- Track complaint-to-churn correlation

## Production Considerations
- Automate theme classification with an NLP model for incoming complaints
- Set up real-time dashboards for complaint volume, severity, and escalation rates
- Define SLAs by severity: Critical < 24h, High < 48h, etc.
- Route complaints by detected theme to the appropriate team automatically

## Common Mistakes
- Treating all complaints equally without severity weighting
- Focusing on complaint count rather than business impact (revenue at risk, churn probability)
- Not closing the loop: analyzing complaints without feeding insights back to product/ops teams
- Ignoring low-volume but high-severity themes in favor of high-volume low-severity ones

## Mini Challenge / Exercises
1. Which product has the highest escalation rate? Is it also the most complained-about product?
2. Calculate the "cost of complaints" assuming each escalated complaint costs $50 in support time.
3. Build a simple complaint priority score: Severity(1-4) × Escalation(0/1) × Resolution_Days.
4. Which channel has the fastest average resolution? Why might that be?

## Final Summary / Key Takeaways
- Systematic complaint analysis reveals actionable patterns that reactive handling misses
- Theme × Severity analysis prioritizes which problems to fix first
- Escalation rate is a strong signal for customer frustration intensity
- Product-level complaint profiles identify quality and process failures
- The highest-impact improvement is often fixing the root cause of the #1 complaint theme